In [35]:
from pathlib import Path
import polars as pl
from regex_generator import words_to_compact_regex

path = Path("/workspaces/example/data/projects/ricu/inst/extdata/config/concept-dict.json")
df = pl.read_json(path)
d_items_df = pl.scan_csv("/workspaces/example/data/physionet.org/files/mimiciv/3.1/icu/d_items.csv.gz")
d_items_df = d_items_df.with_columns((pl.col("itemid").cast(pl.Utf8) + "//" + pl.col("label")).alias("code"))
d_labitems_df = pl.scan_csv("/workspaces/example/data/physionet.org/files/mimiciv/3.1/hosp/d_labitems.csv.gz")
d_labitems_df = d_labitems_df.with_columns((pl.col("itemid").cast(pl.Utf8) + "//" + pl.col("label")).alias("code"))


In [36]:
entries = []
for column in df.columns:
    if (srcs := df[column][0].get("sources")) and (miiv := srcs.get("miiv")):
        parent_folder = df[column][0].get("category")
        name = df[column][0].get("description")
        extension_columns = {
            "dataset": " ",
            "table": " ",
        }
        if ids := miiv[0].get("ids"):
            if isinstance(ids, str) or (isinstance(ids, list) and isinstance(ids[0], str)):
                continue
            ids: list[str] = ids if isinstance(ids, list) else [ids]

            # TODO: get labels and write into regex
            filtered_items = d_items_df.filter(pl.col("itemid").is_in(ids)).select("code").collect()
            filtered_labitems = d_labitems_df.filter(pl.col("itemid").is_in(ids)).select("code").collect()

            regex = list(filtered_items.to_dict()["code"]) + list(filtered_labitems.to_dict()["code"])
        
        pattern = []
        for miiv_e in miiv:
            entry = {key:miiv_e.get(key) for key in miiv[0].keys()}
            if (not isinstance(ids, bool)) and (ids := entry.get("ids")) and (not entry.get("regex")):
                if isinstance(ids, str) or (isinstance(ids, list) and isinstance(ids[0], str)):
                    continue
                ids: list[str] = ids if isinstance(ids, list) else [ids]

                filtered_items = d_items_df.filter(pl.col("itemid").is_in(ids)).select("code").collect()
                filtered_labitems = d_labitems_df.filter(pl.col("itemid").is_in(ids)).select("code").collect()

                # regex = list(filtered_items.to_dict()["label"]) + list(filtered_labitems.to_dict()["label"])
                entry["regex"] = "|".join(list(filtered_items.to_dict()["code"]) + list(filtered_labitems.to_dict()["code"]))
                print(words_to_compact_regex(list(filtered_items.to_dict()["code"]) + list(filtered_labitems.to_dict()["code"])))

            if entry.get("regex"):
                entry["regex"] = "(" + entry["regex"] + ")"
            pattern.append(entry)

        mapping = {
            "dataset": "miiv",
            # "pattern":  [{key:miiv_e.get(key) for key in miiv[0].keys()} for miiv_e in miiv],
            "pattern":  pattern,
        }

        entries += {
            "parent_folder": parent_folder,
            "name": name,
            "extension_columns": extension_columns,
            "mapping":mapping,
        },       

new_entries: dict[str, list] = {}
for entry in entries:
    if not new_entries.get(entry["parent_folder"]):
        new_entries[entry["parent_folder"]] = []
    new_entries[entry["parent_folder"]].append(entry)
entries = new_entries

(51274//PT)
(50911//Creatine Kinase, MB Isoenzyme)
(51006//Urea Nitrogen)
(221653//Dobutamine)
(50808//Free Calcium)
(221906//Norepinephrine)
(50910//Creatine Kinase \(CK\))
(22(02(27//Arterial O2 Saturation|77//O2 saturation pulseoxymetry)|6253//SpO2 Desat Limit))
(50817//Oxygen Saturation)
(50885//Bilirubin, Total)
(51002//Troponin I)
(50883//Bilirubin, Direct)
(221289//Epinephrine)
(22579(2//Invasive Ventilation|4//Non\-invasive Ventilation))
(50(809//Glucose|931//Glucose))
(51288//Sedimentation Rate)
(50821//pO2)
(51214//Fibrinogen, Functional)
(50971//Potassium)
(50970//Phosphate)
(50818//pCO2)
(228640//EtCO2)
(51144//Bands)
(221749//Phenylephrine)
(51277//RDW)
(50882//Bicarbonate)
(2232(58//Insulin \- Regular|60//Insulin \- Glargine))
(50902//Chloride)
(221906//Norepinephrine)
(51275//PTT)
(22(65(5(7//R Ureteral Stent|8//L Ureteral Stent|9//Foley)|6(0//Void|1//Condom Cath|3//Suprapubic|4//R Nephrostomy|5//L Nephrostomy|6//Urine and GU Irrigant Out|7//Straight Cath)|84//Ileocondui

In [37]:
import json

with open("data.json", "w", encoding="utf-8") as f:
    json.dump(entries, f, indent=2, ensure_ascii=False)

In [38]:
for key in entries.keys():
    print(key)

hematology
chemistry
demographics
medications
blood gas
respiratory
outcome
vitals
output
microbiology
neurological


In [39]:
#  MIMIC.chemistry is subset of OpenICU.observation 
for entry in entries["chemistry"]:
    print(entry["name"])

creatine kinase MB
blood urea nitrogen
creatine kinase
total bilirubin
troponin I
bilirubin direct
glucose
potassium
phosphate
bicarbonate
chloride
aspartate aminotransferase
magnesium
alanine aminotransferase
troponin t
calcium
sodium
albumin
creatinine
alkaline phosphatase
C-reactive protein


In [40]:
d_labitems_df.filter(pl.col("label").str.contains(".*?(Urine Creatinine)")).head(10).collect()

itemid,label,fluid,category,code
i64,str,str,str,str
51106,"""Urine Creatinine""","""Urine""","""Chemistry""","""51106//Urine Creatinine"""


In [41]:
for entry in entries["chemistry"]:
    print(entry["name"], entry["mapping"]["pattern"][0]["regex"])

creatine kinase MB (50911//Creatine Kinase, MB Isoenzyme)
blood urea nitrogen (51006//Urea Nitrogen)
creatine kinase (50910//Creatine Kinase (CK))
total bilirubin (50885//Bilirubin, Total)
troponin I (51002//Troponin I)
bilirubin direct (50883//Bilirubin, Direct)
glucose (50809//Glucose|50931//Glucose)
potassium (50971//Potassium)
phosphate (50970//Phosphate)
bicarbonate (50882//Bicarbonate)
chloride (50902//Chloride)
aspartate aminotransferase (50878//Asparate Aminotransferase (AST))
magnesium (50960//Magnesium)
alanine aminotransferase (50861//Alanine Aminotransferase (ALT))
troponin t (51003//Troponin T)
calcium (50893//Calcium, Total)
sodium (50983//Sodium)
albumin (50862//Albumin)
creatinine (50912//Creatinine)
alkaline phosphatase (50863//Alkaline Phosphatase)
C-reactive protein (50889//C-Reactive Protein)


In [42]:
for key in entries.keys():
    print(key)
    for entry in entries[key]:
        print(entry)

hematology
{'parent_folder': 'hematology', 'name': 'prothrombine time', 'extension_columns': {'dataset': ' ', 'table': ' '}, 'mapping': {'dataset': 'miiv', 'pattern': [{'ids': 51274, 'table': 'labevents', 'sub_var': 'itemid', 'regex': '(51274//PT)'}]}}
{'parent_folder': 'hematology', 'name': 'erythrocyte sedimentation rate', 'extension_columns': {'dataset': ' ', 'table': ' '}, 'mapping': {'dataset': 'miiv', 'pattern': [{'ids': 51288, 'table': 'labevents', 'sub_var': 'itemid', 'regex': '(51288//Sedimentation Rate)'}]}}
{'parent_folder': 'hematology', 'name': 'fibrinogen', 'extension_columns': {'dataset': ' ', 'table': ' '}, 'mapping': {'dataset': 'miiv', 'pattern': [{'ids': 51214, 'table': 'labevents', 'sub_var': 'itemid', 'regex': '(51214//Fibrinogen, Functional)'}]}}
{'parent_folder': 'hematology', 'name': 'band form neutrophils', 'extension_columns': {'dataset': ' ', 'table': ' '}, 'mapping': {'dataset': 'miiv', 'pattern': [{'ids': 51144, 'table': 'labevents', 'sub_var': 'itemid', 'r